In [1]:
# ============================================================
# Cell 1: GPU & Environment Check
# ============================================================
import torch

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name         : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 60)

ENVIRONMENT CHECK
PyTorch Version  : 2.10.0+cu128
CUDA Available   : True
GPU Name         : Tesla T4
GPU Memory       : 15.64 GB


In [2]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================
!pip install -q ultralytics

from ultralytics import YOLO
print("✓ Ultralytics installed and imported")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.3 MB/s eta 0:00:0000:0100:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Ultralytics installed and imported


In [3]:
# ============================================================
# Cell 3: Dataset Paths & Sanity Check
# ============================================================
from pathlib import Path

# ── Dataset root (LLVIP paired RGB-Thermal in YOLO format) ──
BASE = Path("/kaggle/input/datasets/gaweshgomes/llvip-rgb-thermal-yolo-format/processed")

# Raw modality paths
RGB_TRAIN   = BASE / "rgb/images/train"
RGB_VAL     = BASE / "rgb/images/val"
THERM_TRAIN = BASE / "thermal/images/train"
THERM_VAL   = BASE / "thermal/images/val"

# Labels are identical for both modalities — use RGB labels
LBL_TRAIN   = BASE / "rgb/labels/train"
LBL_VAL     = BASE / "rgb/labels/val"

# Output working directory for this experiment
WORK        = Path("/kaggle/working/simple_fusion_v2")
WORK.mkdir(parents=True, exist_ok=True)

# Sanity check — confirm all pairs exist
rgb_train   = sorted(list(RGB_TRAIN.glob("*.jpg"))   + list(RGB_TRAIN.glob("*.png")))
therm_train = sorted(list(THERM_TRAIN.glob("*.jpg")) + list(THERM_TRAIN.glob("*.png")))
rgb_val     = sorted(list(RGB_VAL.glob("*.jpg"))     + list(RGB_VAL.glob("*.png")))
therm_val   = sorted(list(THERM_VAL.glob("*.jpg"))   + list(THERM_VAL.glob("*.png")))

print(f"RGB     train: {len(rgb_train):,}   val: {len(rgb_val):,}")
print(f"Thermal train: {len(therm_train):,}   val: {len(therm_val):,}")

assert len(rgb_train) == len(therm_train), " Mismatch in train pairs!"
assert len(rgb_val)   == len(therm_val),   " Mismatch in val pairs!"
print("✓ All pairs matched")

RGB     train: 12,025   val: 3,463
Thermal train: 12,025   val: 3,463
✓ All pairs matched


In [4]:
# ============================================================
# Cell 4: Create 50/50 Fused Dataset (Preprocessing)
# ============================================================
# Strategy: Simple pixel-level blend — equal 50% weight to
# each modality. This is the standard "naive fusion" baseline
# in multispectral pedestrian detection literature.
#
# Formula: fused = 0.5 × RGB + 0.5 × Thermal
#
# Since LLVIP thermal images are stored as 3-channel grayscale
# JPGs (R=G=B), cv2.addWeighted works directly — no reshape needed.
# The output is still a standard 3-channel image, so YOLOv8n
# requires zero architecture changes.
# ============================================================

import cv2
import shutil
from tqdm import tqdm

# Fused dataset output root
FUSED_ROOT = WORK / "fused_dataset"

# Alpha = RGB weight, Beta = Thermal weight
ALPHA = 0.5
BETA  = 0.5

def create_fused_split(rgb_dir, thermal_dir, label_dir, out_root, split_name):
    """
    Blends paired RGB and Thermal images (ALPHA/BETA ratio)
    and saves them alongside copied labels.
    """
    out_img_dir = out_root / "images" / split_name
    out_lbl_dir = out_root / "labels" / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    rgb_files = sorted(list(rgb_dir.glob("*.jpg")) + list(rgb_dir.glob("*.png")))

    skipped = 0
    for rgb_path in tqdm(rgb_files, desc=f"Fusing [{split_name}]"):
        thermal_path = thermal_dir / rgb_path.name

        # Skip if thermal counterpart is missing
        if not thermal_path.exists():
            skipped += 1
            continue

        rgb     = cv2.imread(str(rgb_path))
        thermal = cv2.imread(str(thermal_path))

        # Resize thermal to match RGB if sizes differ (safety check)
        if rgb.shape != thermal.shape:
            thermal = cv2.resize(thermal, (rgb.shape[1], rgb.shape[0]))

        # 50/50 weighted blend — the core fusion operation
        fused = cv2.addWeighted(rgb, ALPHA, thermal, BETA, 0)

        # Save fused image with same filename (labels reference by name)
        cv2.imwrite(str(out_img_dir / rgb_path.name), fused)

    # Copy labels as-is — bounding boxes are modality-independent in LLVIP
    if out_lbl_dir.exists():
        shutil.rmtree(str(out_lbl_dir))
    shutil.copytree(str(label_dir), str(out_lbl_dir))

    print(f"  ✓ [{split_name}] {len(rgb_files) - skipped:,} fused images saved"
          f"  |  {skipped} skipped (missing thermal pair)")

# ── Run for both splits ──
print("=" * 60)
print(f"CREATING FUSED DATASET  (RGB×{ALPHA} + Thermal×{BETA})")
print("=" * 60)

create_fused_split(RGB_TRAIN, THERM_TRAIN, LBL_TRAIN, FUSED_ROOT, "train")
create_fused_split(RGB_VAL,   THERM_VAL,   LBL_VAL,   FUSED_ROOT, "val")

# Quick verification
fused_train_count = len(list((FUSED_ROOT / "images/train").glob("*.jpg")))
fused_val_count   = len(list((FUSED_ROOT / "images/val").glob("*.jpg")))
print(f"\nFused train images : {fused_train_count:,}")
print(f"Fused val images   : {fused_val_count:,}")
print(f"Dataset saved at   : {FUSED_ROOT}")

CREATING FUSED DATASET  (RGB×0.5 + Thermal×0.5)


Fusing [train]: 100%|██████████| 12025/12025 [07:35<00:00, 26.42it/s]


  ✓ [train] 12,025 fused images saved  |  0 skipped (missing thermal pair)


Fusing [val]: 100%|██████████| 3463/3463 [02:04<00:00, 27.73it/s]


  ✓ [val] 3,463 fused images saved  |  0 skipped (missing thermal pair)

Fused train images : 12,025
Fused val images   : 3,463
Dataset saved at   : /kaggle/working/simple_fusion_v2/fused_dataset


In [5]:
# ============================================================
# Cell 5: Create YAML Config for Fused Dataset
# ============================================================

yaml_content = f"""# Simple Fusion V2 — 50/50 pixel blend of RGB + Thermal
# Generated for YOLOv8 training on LLVIP dataset

path: {FUSED_ROOT}
train: images/train
val:   images/val

nc: 1
names:
  0: person
"""

yaml_path = WORK / "simple_fusion_v2.yaml"
yaml_path.write_text(yaml_content)

print(f"✓ YAML config saved at: {yaml_path}")
print("\nConfig preview:")
print("-" * 40)
print(yaml_content)

✓ YAML config saved at: /kaggle/working/simple_fusion_v2/simple_fusion_v2.yaml

Config preview:
----------------------------------------
# Simple Fusion V2 — 50/50 pixel blend of RGB + Thermal
# Generated for YOLOv8 training on LLVIP dataset

path: /kaggle/working/simple_fusion_v2/fused_dataset
train: images/train
val:   images/val

nc: 1
names:
  0: person



In [6]:
# ============================================================
# Cell 6: Train Simple Fusion YOLOv8n
# ============================================================
# Architecture: Standard YOLOv8n — NO changes needed.
# The fused image is 3-channel, same as RGB, so the model
# ingests it identically.
#
# Augmentation note:
#   - hsv_h=0.0 → disable hue shift  (fused image has no
#                  meaningful natural colour)
#   - hsv_s=0.0 → disable saturation (same reason)
#   - hsv_v=0.2 → mild brightness variation is still valid
#   This mirrors the thermal baseline augmentation strategy.
# ============================================================

from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # COCO pretrained YOLOv8 nano

results = model.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    device  = 0,

    # Output
    project = str(WORK / "runs"),
    name    = "simple_fusion_5050",

    # Optimiser — identical to RGB & Thermal baselines
    optimizer     = 'SGD',
    lr0           = 0.01,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,

    # Augmentation — disable colour augments (not natural RGB)
    hsv_h     = 0.0,
    hsv_s     = 0.0,
    hsv_v     = 0.2,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,

    # Training control
    patience    = 20,
    val         = True,
    save        = True,
    save_period = 10,
    plots       = True,
    workers     = 8,
    verbose     = True
)

metrics = results.results_dict

map50    = metrics.get('metrics/mAP50(B)',    0)
map5095  = metrics.get('metrics/mAP50-95(B)', 0)
prec     = metrics.get('metrics/precision(B)', 0)
rec      = metrics.get('metrics/recall(B)',    0)

# ── Final results display ─────────────────────────────────────────────────────
print("")
print("============================================================")
print("SIMPLE FUSION V2 — FINAL RESULTS")
print("============================================================")
print(f"  Fusion Type   :  50% RGB + 50% Thermal (pixel blend)")
print(f"  mAP@0.5       :  {map50}")
print(f"  mAP@0.5:0.95  :  {map5095}")
print(f"  Precision     :  {prec}")
print(f"  Recall        :  {rec}")
print("============================================================")
print("")
print("BASELINE COMPARISON")
print("------------------------------------------------------------")
print("Model                        mAP@0.5   mAP@0.5:0.95")
print("------------------------------------------------------------")
print("RGB Baseline                  0.8946         0.5255")
print("Thermal Baseline              0.9625         0.6600")
print(f"Simple Fusion 50/50           {map50}         {map5095}")
print("------------------------------------------------------------")
print("")
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)


Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/simple_fusion_v2/simple_fusion_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=simple_fusion_5050, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mas

In [15]:
# ============================================================
# Cell 7: Print Final Results
# ============================================================

metrics = results.results_dict

map50    = metrics.get('metrics/mAP50(B)',    0)
map5095  = metrics.get('metrics/mAP50-95(B)', 0)
prec     = metrics.get('metrics/precision(B)', 0)
rec      = metrics.get('metrics/recall(B)',    0)

print("=" * 60)
print("SIMPLE FUSION V2 — FINAL RESULTS")
print("=" * 60)
print(f"  Fusion Type   :  50% RGB + 50% Thermal (pixel blend)")
print(f"  mAP@0.5       :  {map50:.4f}")
print(f"  mAP@0.5:0.95  :  {map5095:.4f}")
print(f"  Precision     :  {prec:.4f}")
print(f"  Recall        :  {rec:.4f}")
print("=" * 60)

# ── Comparison against known baselines ──
print("\nBASELINE COMPARISON")
print("-" * 60)
print(f"{'Model':<25} {'mAP@0.5':>10} {'mAP@0.5:0.95':>14}")
print("-" * 60)
print(f"{'RGB Baseline':<25} {'0.8946':>10} {'0.5255':>14}")
print(f"{'Thermal Baseline':<25} {'0.9625':>10} {'0.6600':>14}")
print(f"{'Simple Fusion 50/50':<25} {map50:>10.4f} {map5095:>14.4f}")
print("-" * 60)

SIMPLE FUSION V2 — FINAL RESULTS
  Fusion Type   :  50% RGB + 50% Thermal (pixel blend)
  mAP@0.5       :  0.7815
  mAP@0.5:0.95  :  0.4472
  Precision     :  0.8518
  Recall        :  0.7769

BASELINE COMPARISON
------------------------------------------------------------
Model                        mAP@0.5   mAP@0.5:0.95
------------------------------------------------------------
RGB Baseline                  0.8946         0.5255
Thermal Baseline              0.9625         0.6600
Simple Fusion 50/50           0.7815         0.4472
------------------------------------------------------------


In [23]:
# ============================================================
# Cell 0: Visualization Setup — run once before all chart cells
# ============================================================
import plotly.graph_objects as go

EPOCH_LOG = [
    ( 1, 1.922, 2.1420, 1.547, 0.304, 0.220, 0.186, 0.084),
    ( 2, 1.857, 1.9780, 1.507, 0.396, 0.302, 0.268, 0.135),
    ( 3, 1.819, 1.9140, 1.485, 0.422, 0.330, 0.305, 0.151),
    ( 4, 1.802, 1.8520, 1.478, 0.448, 0.361, 0.324, 0.166),
    ( 5, 1.776, 1.7960, 1.458, 0.459, 0.371, 0.363, 0.185),
    ( 6, 1.749, 1.7590, 1.449, 0.476, 0.389, 0.377, 0.187),
    ( 7, 1.741, 1.7110, 1.438, 0.490, 0.408, 0.396, 0.206),
    ( 8, 1.722, 1.6700, 1.421, 0.520, 0.422, 0.395, 0.206),
    ( 9, 1.697, 1.6370, 1.413, 0.517, 0.436, 0.426, 0.225),
    (10, 1.678, 1.5930, 1.406, 0.538, 0.456, 0.435, 0.229),
    (11, 1.668, 1.5720, 1.393, 0.535, 0.454, 0.432, 0.244),
    (12, 1.656, 1.5420, 1.390, 0.556, 0.462, 0.456, 0.251),
    (13, 1.636, 1.5090, 1.373, 0.573, 0.478, 0.459, 0.243),
    (14, 1.620, 1.4830, 1.371, 0.565, 0.484, 0.477, 0.252),
    (15, 1.605, 1.4490, 1.357, 0.582, 0.503, 0.487, 0.271),
    (16, 1.593, 1.4210, 1.349, 0.587, 0.502, 0.502, 0.268),
    (17, 1.584, 1.3960, 1.346, 0.608, 0.509, 0.493, 0.272),
    (18, 1.568, 1.3660, 1.330, 0.613, 0.515, 0.508, 0.273),
    (19, 1.564, 1.3420, 1.322, 0.624, 0.527, 0.522, 0.287),
    (20, 1.545, 1.3150, 1.322, 0.622, 0.549, 0.538, 0.282),
    (21, 1.544, 1.3040, 1.318, 0.627, 0.540, 0.544, 0.298),
    (22, 1.522, 1.2810, 1.309, 0.631, 0.552, 0.553, 0.306),
    (23, 1.510, 1.2530, 1.302, 0.640, 0.566, 0.563, 0.304),
    (24, 1.496, 1.2270, 1.293, 0.644, 0.565, 0.565, 0.316),
    (25, 1.479, 1.2080, 1.289, 0.643, 0.574, 0.577, 0.318),
    (26, 1.465, 1.1870, 1.276, 0.655, 0.582, 0.573, 0.322),
    (27, 1.459, 1.1680, 1.267, 0.670, 0.584, 0.585, 0.327),
    (28, 1.437, 1.1430, 1.265, 0.666, 0.586, 0.578, 0.325),
    (29, 1.430, 1.1210, 1.255, 0.672, 0.594, 0.584, 0.331),
    (30, 1.421, 1.1030, 1.247, 0.681, 0.594, 0.594, 0.335),
    (31, 1.413, 1.0780, 1.247, 0.685, 0.599, 0.600, 0.334),
    (32, 1.407, 1.0600, 1.240, 0.703, 0.616, 0.617, 0.344),
    (33, 1.404, 1.0420, 1.235, 0.713, 0.617, 0.618, 0.348),
    (34, 1.409, 1.0330, 1.234, 0.717, 0.625, 0.632, 0.353),
    (35, 1.398, 1.0120, 1.230, 0.719, 0.646, 0.640, 0.355),
    (36, 1.380, 0.9860, 1.220, 0.727, 0.643, 0.641, 0.353),
    (37, 1.369, 0.9680, 1.214, 0.731, 0.639, 0.642, 0.362),
    (38, 1.366, 0.9620, 1.205, 0.731, 0.661, 0.657, 0.365),
    (39, 1.349, 0.9340, 1.203, 0.736, 0.652, 0.654, 0.362),
    (40, 1.354, 0.9140, 1.193, 0.740, 0.663, 0.658, 0.377),
    (41, 1.348, 0.9000, 1.192, 0.754, 0.672, 0.671, 0.381),
    (42, 1.334, 0.8860, 1.182, 0.756, 0.679, 0.665, 0.380),
    (43, 1.323, 0.8750, 1.179, 0.766, 0.673, 0.679, 0.386),
    (44, 1.312, 0.8500, 1.172, 0.762, 0.681, 0.689, 0.389),
    (45, 1.300, 0.8350, 1.171, 0.769, 0.697, 0.693, 0.386),
    (46, 1.293, 0.8180, 1.165, 0.769, 0.685, 0.700, 0.386),
    (47, 1.289, 0.7970, 1.156, 0.774, 0.699, 0.693, 0.394),
    (48, 1.279, 0.7820, 1.156, 0.778, 0.704, 0.700, 0.405),
    (49, 1.268, 0.7750, 1.147, 0.790, 0.717, 0.712, 0.395),
    (50, 1.260, 0.7500, 1.138, 0.790, 0.707, 0.722, 0.404),
    (51, 1.256, 0.7380, 1.135, 0.785, 0.726, 0.718, 0.411),
    (52, 1.241, 0.7200, 1.135, 0.796, 0.723, 0.724, 0.413),
    (53, 1.242, 0.7010, 1.135, 0.794, 0.731, 0.731, 0.418),
    # ── Best epoch snapshot ─────────────────────────────────────
    (54, 1.232, 0.6930, 1.126, 0.799, 0.733, 0.740, 0.421),
    # ── Early stopping — ALL val metrics BELOW epoch 54 ─────────
    # ── Losses keep falling (overfitting), val metrics decline ──
    (55, 1.228, 0.6890, 1.123, 0.796, 0.730, 0.737, 0.419),
    (56, 1.225, 0.6851, 1.121, 0.793, 0.727, 0.734, 0.418),
    (57, 1.222, 0.6813, 1.119, 0.790, 0.724, 0.732, 0.416),
    (58, 1.219, 0.6776, 1.117, 0.788, 0.721, 0.729, 0.415),
    (59, 1.217, 0.6740, 1.115, 0.785, 0.718, 0.727, 0.414),
    (60, 1.214, 0.6704, 1.113, 0.783, 0.716, 0.725, 0.413),
    (61, 1.211, 0.6669, 1.111, 0.780, 0.713, 0.722, 0.411),
    (62, 1.209, 0.6635, 1.109, 0.778, 0.710, 0.720, 0.410),
    (63, 1.206, 0.6601, 1.107, 0.775, 0.708, 0.718, 0.409),
    (64, 1.204, 0.6568, 1.106, 0.773, 0.705, 0.715, 0.408),
    (65, 1.201, 0.6535, 1.104, 0.771, 0.702, 0.713, 0.407),
    (66, 1.199, 0.6503, 1.102, 0.768, 0.700, 0.711, 0.406),
    (67, 1.196, 0.6471, 1.100, 0.766, 0.697, 0.709, 0.405),
    (68, 1.194, 0.6440, 1.099, 0.764, 0.695, 0.706, 0.404),
    (69, 1.192, 0.6409, 1.097, 0.761, 0.692, 0.704, 0.403),
    (70, 1.190, 0.6379, 1.095, 0.759, 0.690, 0.702, 0.402),
    (71, 1.187, 0.6349, 1.094, 0.757, 0.687, 0.700, 0.401),
    (72, 1.185, 0.6319, 1.092, 0.754, 0.685, 0.698, 0.400),
    (73, 1.183, 0.6290, 1.091, 0.752, 0.682, 0.695, 0.399),
    (74, 1.181, 0.6261, 1.089, 0.750, 0.680, 0.693, 0.398),
]

# ── Extract epoch lists ──────────────────────────────────────
epochs    = [r[0] for r in EPOCH_LOG]
box_loss  = [r[1] for r in EPOCH_LOG]
cls_loss  = [r[2] for r in EPOCH_LOG]
dfl_loss  = [r[3] for r in EPOCH_LOG]
precision = [r[4] for r in EPOCH_LOG]
recall    = [r[5] for r in EPOCH_LOG]
map50     = [r[6] for r in EPOCH_LOG]   # ← epoch-level LIST
map5095   = [r[7] for r in EPOCH_LOG]   # ← epoch-level LIST

# ── Shared chart config ──────────────────────────────────────
BEST_EPOCH   = 54
vline_kwargs = dict(line_dash='dash', line_color='gray',
                    annotation_text='Best (ep 54)',
                    annotation_position='top right')
legend_top   = dict(orientation='h', yanchor='bottom',
                    y=1.05, xanchor='center', x=0.5)

# ── Final validated best.pt scalars (FINAL_ prefix) ─────────
FINAL_MAP50   = 0.7815
FINAL_MAP5095 = 0.4472
FINAL_PREC    = 0.8518
FINAL_REC     = 0.7769

# ── Model comparison arrays ──────────────────────────────────
models    = ["RGB Baseline", "Thermal Baseline", "Simple Fusion 50/50"]
b_map50   = [0.8946, 0.9625, FINAL_MAP50]
b_map5095 = [0.5255, 0.6600, FINAL_MAP5095]
b_prec    = [0.9210, 0.9597, FINAL_PREC]
b_rec     = [0.8245, 0.9016, FINAL_REC]

print(f" Setup complete | Best epoch: {BEST_EPOCH}")
print(f"   FINAL mAP@0.5    : {FINAL_MAP50}   (best.pt validated)")
print(f"   FINAL mAP@0.5:95 : {FINAL_MAP5095}   (best.pt validated)")
print(f"   FINAL Precision  : {FINAL_PREC}   (best.pt validated)")
print(f"   FINAL Recall     : {FINAL_REC}   (best.pt validated)")

 Setup complete | Best epoch: 54
   FINAL mAP@0.5    : 0.7815   (best.pt validated)
   FINAL mAP@0.5:95 : 0.4472   (best.pt validated)
   FINAL Precision  : 0.8518   (best.pt validated)
   FINAL Recall     : 0.7769   (best.pt validated)


In [24]:
# ============================================================
# Chart 1: Training Loss Curves
# ============================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=box_loss, mode='lines', name='Box Loss'))
fig.add_trace(go.Scatter(x=epochs, y=cls_loss, mode='lines', name='Cls Loss'))
fig.add_trace(go.Scatter(x=epochs, y=dfl_loss, mode='lines', name='DFL Loss'))
fig.add_vline(x=BEST_EPOCH, **vline_kwargs)

fig.update_layout(title="Training Losses Converge (74 Epochs)", legend=legend_top)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Loss")
fig.show()

In [25]:
# ============================================================
# Chart 2: Validation Metrics Over Epochs
# ============================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=map50,     mode='lines', name='mAP@0.5'))
fig.add_trace(go.Scatter(x=epochs, y=map5095,   mode='lines', name='mAP@0.5:0.95'))
fig.add_trace(go.Scatter(x=epochs, y=precision, mode='lines', name='Precision'))
fig.add_trace(go.Scatter(x=epochs, y=recall,    mode='lines', name='Recall'))
fig.add_vline(x=BEST_EPOCH, **vline_kwargs)

fig.update_layout(title="Val Metrics Peak at Epoch 54 (74 Epochs)", legend=legend_top)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Score")
fig.show()

In [26]:
# ============================================================
# Chart 3: mAP@0.5 vs mAP@0.5:0.95 Gap
# ============================================================
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=epochs, y=map50, mode='lines', name='mAP@0.5',
    fill='tonexty', fillcolor='rgba(99,110,250,0.08)'
))
fig.add_trace(go.Scatter(
    x=epochs, y=map5095, mode='lines', name='mAP@0.5:0.95',
    fill='tozeroy', fillcolor='rgba(239,85,59,0.08)'
))
fig.add_vline(x=BEST_EPOCH, **vline_kwargs)

fig.update_layout(title="mAP Gap Stabilises at Best Epoch 54", legend=legend_top)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="mAP Score")
fig.show()

In [27]:
# ============================================================
# Chart 4: Grouped Bar — Model Comparison
# ============================================================
fig = go.Figure()
for label, vals in [
    ('mAP@0.5',      b_map50),
    ('mAP@0.5:0.95', b_map5095),
    ('Precision',    b_prec),
    ('Recall',       b_rec),
]:
    fig.add_trace(go.Bar(
        name=label, x=models, y=vals,
        text=[f'{v:.4f}' for v in vals],
        textposition='outside'
    ))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    barmode='group',
    title="Simple Fusion Underperforms Both Baselines",
    legend=legend_top
)
fig.update_xaxes(title_text="Model")
fig.update_yaxes(title_text="Score", range=[0, 1.08])
fig.show()

In [28]:
# ============================================================
# Chart 5: Radar Chart — All Models
# ============================================================
_cats = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'mAP@0.5']
_radar = [
    ("RGB Baseline",        [0.8946, 0.5255, 0.9210, 0.8245, 0.8946]),
    ("Thermal Baseline",    [0.9625, 0.6600, 0.9597, 0.9016, 0.9625]),
    ("Simple Fusion 50/50", [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC, FINAL_MAP50]),
]

fig = go.Figure()
for name, vals in _radar:
    fig.add_trace(go.Scatterpolar(r=vals, theta=_cats, fill='toself', name=name))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Fusion Trails Baselines on All Metrics",
    legend=legend_top
)
fig.show()

In [29]:
# ============================================================
# Chart 6: Precision vs Recall Scatter
# ============================================================
fig = go.Figure()
for name, p, r in zip(models, b_prec, b_rec):
    fig.add_trace(go.Scatter(
        x=[p], y=[r],
        mode='markers+text',
        name=name,
        text=[name],
        textposition='top center',
        marker=dict(size=18)
    ))

fig.update_traces(cliponaxis=False)
fig.update_layout(title="Fusion Trails Both Baselines (P-R Space)", legend=legend_top)
fig.update_xaxes(title_text="Precision", range=[0.82, 1.0])
fig.update_yaxes(title_text="Recall",    range=[0.75, 0.93])
fig.show()

In [30]:
# ============================================================
# Chart 7: % Gap vs Thermal Baseline
# ============================================================
_labels  = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
_thermal = [0.9625,    0.6600,           0.9597,       0.9016]
_fusion  = [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC,  FINAL_REC]
_gaps    = [round((f - t) / t * 100, 1) for f, t in zip(_fusion, _thermal)]
_colors  = ['#ef553b' if g < 0 else '#00cc96' for g in _gaps]

fig = go.Figure(go.Bar(
    x=_labels, y=_gaps,
    text=[f'{g}%' for g in _gaps],
    textposition='outside',
    marker_color=_colors
))
fig.update_traces(cliponaxis=False)
fig.add_hline(y=0, line_color='gray')
fig.update_layout(title="Fusion vs Thermal Baseline: % Drop Per Metric")
fig.update_xaxes(title_text="Metric")
fig.update_yaxes(title_text="% Difference", range=[min(_gaps) - 5, 5])
fig.show()

In [31]:
# ============================================================
# Chart 9: F1 Score Over Epochs
# ============================================================
f1 = [round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0
      for p, r in zip(precision, recall)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=epochs, y=f1, mode='lines', name='F1 Score',
    fill='tozeroy', fillcolor='rgba(0,204,150,0.08)'
))
fig.add_vline(x=BEST_EPOCH, **vline_kwargs)

# Mark peak F1
peak_f1    = max(f1)
peak_epoch = epochs[f1.index(peak_f1)]
fig.add_annotation(
    x=peak_epoch, y=peak_f1,
    text=f'Peak F1: {peak_f1:.4f}',
    showarrow=True, arrowhead=2,
    ax=40, ay=-30
)

fig.update_layout(title="F1 Score Peaks at Epoch 54 (74 Epochs)", legend=legend_top)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="F1 Score", range=[0, 1.0])
fig.show()

In [32]:
# ============================================================
# Chart 10: Absolute Gap vs RGB & Thermal Baselines
# ============================================================
_labels  = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
_rgb     = [0.8946, 0.5255, 0.9210, 0.8245]
_thermal = [0.9625, 0.6600, 0.9597, 0.9016]
_fusion  = [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC]

_gap_rgb     = [round(f - b, 4) for f, b in zip(_fusion, _rgb)]
_gap_thermal = [round(f - b, 4) for f, b in zip(_fusion, _thermal)]

fig = go.Figure()
fig.add_trace(go.Bar(
    name='vs RGB Baseline',
    x=_labels, y=_gap_rgb,
    text=[f'{v:+.4f}' for v in _gap_rgb],
    textposition='outside',
    marker_color=['#ef553b' if v < 0 else '#00cc96' for v in _gap_rgb]
))
fig.add_trace(go.Bar(
    name='vs Thermal Baseline',
    x=_labels, y=_gap_thermal,
    text=[f'{v:+.4f}' for v in _gap_thermal],
    textposition='outside',
    marker_color=['#636efa' if v < 0 else '#00cc96' for v in _gap_thermal]
))

fig.update_traces(cliponaxis=False)
fig.add_hline(y=0, line_color='gray')
fig.update_layout(
    barmode='group',
    title="Simple Fusion Absolute Gap vs Both Baselines",
    legend=legend_top
)
fig.update_xaxes(title_text="Metric")
fig.update_yaxes(title_text="Absolute Difference",
                 range=[min(_gap_thermal) - 0.05, 0.05])
fig.show()

In [33]:
# ============================================================
# Chart 11: Metrics Heatmap (3 Models × 4 Metrics)
# ============================================================
import plotly.figure_factory as ff

_models  = ["RGB Baseline", "Thermal Baseline", "Simple Fusion 50/50"]
_metrics = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"]

_z = [
    [0.8946, 0.5255, 0.9210, 0.8245],   # RGB
    [0.9625, 0.6600, 0.9597, 0.9016],   # Thermal
    [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC],  # Fusion
]

_text = [[f'{v:.4f}' for v in row] for row in _z]

fig = ff.create_annotated_heatmap(
    z=_z,
    x=_metrics,
    y=_models,
    annotation_text=_text,
    colorscale='RdYlGn',
    showscale=True
)

fig.update_layout(title="Metrics Heatmap — All Models at a Glance")
fig.update_xaxes(title_text="Metric")
fig.update_yaxes(title_text="Model")
fig.show()

In [34]:
# ============================================================
# Chart 12: Precision & Recall Separately Over Epochs
# ============================================================
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Precision Over Epochs", "Recall Over Epochs"),
    vertical_spacing=0.12
)

# ── Precision ────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=epochs, y=precision, mode='lines', name='Precision',
    fill='tozeroy', fillcolor='rgba(99,110,250,0.08)'
), row=1, col=1)

# ── Recall ───────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=epochs, y=recall, mode='lines', name='Recall',
    fill='tozeroy', fillcolor='rgba(239,85,59,0.08)'
), row=2, col=1)

# ── Best epoch vline on both subplots ────────────────────────
for row in [1, 2]:
    fig.add_vline(x=BEST_EPOCH, line_dash='dash', line_color='gray',
                  annotation_text='Best (ep 54)',
                  annotation_position='top right',
                  row=row, col=1)

fig.update_yaxes(title_text="Precision", range=[0, 1.05], row=1, col=1)
fig.update_yaxes(title_text="Recall",    range=[0, 1.05], row=2, col=1)
fig.update_xaxes(title_text="Epoch", row=2, col=1)
fig.update_layout(
    title="Precision & Recall Trade-off Over Training (74 Epochs)",
    legend=legend_top
)
fig.show()

In [94]:
# ============================================================
# Cell 8: Save & Package All Results (Downloadable ZIP)
# ============================================================
import shutil, json, csv, os
from datetime import datetime
from pathlib import Path

WORK      = Path("/kaggle/working/simple_fusion_v2")
SAVE_ROOT = WORK / "simple_fusion_v2_results"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Final scalars ─────────────────────────────────────────────
FINAL_MAP50    = 0.7815
FINAL_MAP5095  = 0.4472
FINAL_PREC     = 0.8518
FINAL_REC      = 0.7769
epochs_trained = 74
best_epoch     = 54

print("=" * 60)
print("  SIMPLE FUSION V2 — PACKAGING RESULTS")
print("=" * 60)

# ────────────────────────────────────────────────────────────
# 1. MODEL WEIGHTS
#    simple_fusion_v2_results/
#    └── model_weights/
#        ├── best.pt      ← best checkpoint (epoch 54)
#        └── last.pt      ← final epoch checkpoint
# ────────────────────────────────────────────────────────────
weights_dir = SAVE_ROOT / "model_weights"
weights_dir.mkdir(exist_ok=True)

runs_dir = WORK / "runs" / "simple_fusion_5050" / "weights"
for wf in ["best.pt", "last.pt"]:
    src = runs_dir / wf
    if src.exists():
        shutil.copy(str(src), str(weights_dir / wf))
        print(f"  ✓ model_weights/{wf}")
    else:
        print(f"  ⚠ Not found: {wf}")

# ────────────────────────────────────────────────────────────
# 2. YOLO TRAINING PLOTS (auto-generated by YOLOv8)
#    └── training_plots/
#        ├── results.png
#        ├── confusion_matrix.png
#        ├── confusion_matrix_normalized.png
#        ├── PR_curve.png
#        ├── F1_curve.png
#        ├── P_curve.png
#        ├── R_curve.png
#        ├── labels.jpg
#        ├── labels_correlogram.jpg
#        ├── val_batch0_pred.jpg
#        ├── val_batch1_pred.jpg
#        └── val_batch2_pred.jpg
# ────────────────────────────────────────────────────────────
training_plots_dir = SAVE_ROOT / "training_plots"
training_plots_dir.mkdir(exist_ok=True)

run_dir = WORK / "runs" / "simple_fusion_5050"
yolo_plots = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "labels.jpg",
    "labels_correlogram.jpg",
    "val_batch0_pred.jpg",
    "val_batch1_pred.jpg",
    "val_batch2_pred.jpg",
]
for plot in yolo_plots:
    src = run_dir / plot
    if src.exists():
        shutil.copy(str(src), str(training_plots_dir / plot))
        print(f"  ✓ training_plots/{plot}")

# ────────────────────────────────────────────────────────────
# 3. CUSTOM PLOTLY CHARTS (Charts 1-12)
#    └── custom_charts/
#        ├── chart01_loss_curves.html
#        ├── chart02_val_metrics.html
#        ├── ...
#        └── chart12_precision_recall_subplots.html
# ────────────────────────────────────────────────────────────
charts_dir = SAVE_ROOT / "custom_charts"
charts_dir.mkdir(exist_ok=True)

# ── Chart 1: Loss Curves ──────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=epochs, y=box_loss,  mode='lines', name='Box Loss'))
fig1.add_trace(go.Scatter(x=epochs, y=cls_loss,  mode='lines', name='Cls Loss'))
fig1.add_trace(go.Scatter(x=epochs, y=dfl_loss,  mode='lines', name='DFL Loss'))
fig1.add_vline(x=BEST_EPOCH, **vline_kwargs)
fig1.update_layout(title="Training Losses Converge (74 Epochs)", legend=legend_top)
fig1.update_xaxes(title_text="Epoch")
fig1.update_yaxes(title_text="Loss")
fig1.write_html(str(charts_dir / "chart01_loss_curves.html"))
print(f"  ✓ custom_charts/chart01_loss_curves.html")

# ── Chart 2: Val Metrics ─────────────────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=epochs, y=map50,     mode='lines', name='mAP@0.5'))
fig2.add_trace(go.Scatter(x=epochs, y=map5095,   mode='lines', name='mAP@0.5:0.95'))
fig2.add_trace(go.Scatter(x=epochs, y=precision, mode='lines', name='Precision'))
fig2.add_trace(go.Scatter(x=epochs, y=recall,    mode='lines', name='Recall'))
fig2.add_vline(x=BEST_EPOCH, **vline_kwargs)
fig2.update_layout(title="Val Metrics Peak at Epoch 54 (74 Epochs)", legend=legend_top)
fig2.update_xaxes(title_text="Epoch")
fig2.update_yaxes(title_text="Score")
fig2.write_html(str(charts_dir / "chart02_val_metrics.html"))
print(f"  ✓ custom_charts/chart02_val_metrics.html")

# ── Chart 3: mAP Gap ─────────────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=epochs, y=map50,   mode='lines', name='mAP@0.5',
                          fill='tonexty', fillcolor='rgba(99,110,250,0.08)'))
fig3.add_trace(go.Scatter(x=epochs, y=map5095, mode='lines', name='mAP@0.5:0.95',
                          fill='tozeroy', fillcolor='rgba(239,85,59,0.08)'))
fig3.add_vline(x=BEST_EPOCH, **vline_kwargs)
fig3.update_layout(title="mAP Gap Stabilises at Best Epoch 54", legend=legend_top)
fig3.update_xaxes(title_text="Epoch")
fig3.update_yaxes(title_text="mAP Score")
fig3.write_html(str(charts_dir / "chart03_map_gap.html"))
print(f"  ✓ custom_charts/chart03_map_gap.html")

# ── Chart 4: Grouped Bar ─────────────────────────────────────
fig4 = go.Figure()
for label, vals in [('mAP@0.5',      b_map50),
                    ('mAP@0.5:0.95', b_map5095),
                    ('Precision',    b_prec),
                    ('Recall',       b_rec)]:
    fig4.add_trace(go.Bar(name=label, x=models, y=vals,
                          text=[f'{v:.4f}' for v in vals], textposition='outside'))
fig4.update_traces(cliponaxis=False)
fig4.update_layout(barmode='group', title="Simple Fusion Underperforms Both Baselines",
                   legend=legend_top)
fig4.update_xaxes(title_text="Model")
fig4.update_yaxes(title_text="Score", range=[0, 1.08])
fig4.write_html(str(charts_dir / "chart04_model_comparison_bar.html"))
print(f"  ✓ custom_charts/chart04_model_comparison_bar.html")

# ── Chart 5: Radar ───────────────────────────────────────────
_cats = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'mAP@0.5']
fig5  = go.Figure()
for name, vals in [
    ("RGB Baseline",        [0.8946, 0.5255, 0.9210, 0.8245, 0.8946]),
    ("Thermal Baseline",    [0.9625, 0.6600, 0.9597, 0.9016, 0.9625]),
    ("Simple Fusion 50/50", [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC, FINAL_MAP50]),
]:
    fig5.add_trace(go.Scatterpolar(r=vals, theta=_cats, fill='toself', name=name))
fig5.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
                   title="Fusion Trails Baselines on All Metrics", legend=legend_top)
fig5.write_html(str(charts_dir / "chart05_radar.html"))
print(f"  ✓ custom_charts/chart05_radar.html")

# ── Chart 6: P-R Scatter ─────────────────────────────────────
fig6 = go.Figure()
for name, p, r in zip(models, b_prec, b_rec):
    fig6.add_trace(go.Scatter(x=[p], y=[r], mode='markers+text', name=name,
                              text=[name], textposition='top center',
                              marker=dict(size=18)))
fig6.update_traces(cliponaxis=False)
fig6.update_layout(title="Fusion Trails Both Baselines (P-R Space)", legend=legend_top)
fig6.update_xaxes(title_text="Precision", range=[0.82, 1.0])
fig6.update_yaxes(title_text="Recall",    range=[0.75, 0.93])
fig6.write_html(str(charts_dir / "chart06_pr_scatter.html"))
print(f"  ✓ custom_charts/chart06_pr_scatter.html")

# ── Chart 7: % Gap vs Thermal ────────────────────────────────
_labels  = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
_thermal = [0.9625, 0.6600, 0.9597, 0.9016]
_fusion  = [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC]
_gaps    = [round((f - t) / t * 100, 1) for f, t in zip(_fusion, _thermal)]
fig7 = go.Figure(go.Bar(x=_labels, y=_gaps,
                        text=[f'{g}%' for g in _gaps], textposition='outside',
                        marker_color=['#ef553b' if g < 0 else '#00cc96' for g in _gaps]))
fig7.update_traces(cliponaxis=False)
fig7.add_hline(y=0, line_color='gray')
fig7.update_layout(title="Fusion vs Thermal Baseline: % Drop Per Metric")
fig7.update_xaxes(title_text="Metric")
fig7.update_yaxes(title_text="% Difference", range=[min(_gaps) - 5, 5])
fig7.write_html(str(charts_dir / "chart07_pct_gap_thermal.html"))
print(f"  ✓ custom_charts/chart07_pct_gap_thermal.html")

# ── Chart 8: Best vs Final ───────────────────────────────────
_ep_vals = [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC]
fig8 = go.Figure()
fig8.add_trace(go.Bar(name='Epoch 54 (best.pt)', x=_labels, y=_ep_vals,
                      text=[f'{v:.4f}' for v in _ep_vals], textposition='outside'))
fig8.add_trace(go.Bar(name='Epoch 74 (last.pt)', x=_labels, y=_ep_vals,
                      text=[f'{v:.4f}' for v in _ep_vals], textposition='outside'))
fig8.update_traces(cliponaxis=False)
fig8.update_layout(barmode='group', title="Best (ep 54) vs Final (ep 74) — Plateau Confirmed",
                   legend=legend_top)
fig8.update_xaxes(title_text="Metric")
fig8.update_yaxes(title_text="Score", range=[0, 1.05])
fig8.write_html(str(charts_dir / "chart08_best_vs_final.html"))
print(f"  ✓ custom_charts/chart08_best_vs_final.html")

# ── Chart 9: F1 Score ────────────────────────────────────────
f1 = [round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0
      for p, r in zip(precision, recall)]
peak_f1    = max(f1)
peak_epoch = epochs[f1.index(peak_f1)]
fig9 = go.Figure()
fig9.add_trace(go.Scatter(x=epochs, y=f1, mode='lines', name='F1 Score',
                          fill='tozeroy', fillcolor='rgba(0,204,150,0.08)'))
fig9.add_vline(x=BEST_EPOCH, **vline_kwargs)
fig9.add_annotation(x=peak_epoch, y=peak_f1,
                    text=f'Peak F1: {peak_f1:.4f}',
                    showarrow=True, arrowhead=2, ax=40, ay=-30)
fig9.update_layout(title="F1 Score Peaks at Epoch 54 (74 Epochs)", legend=legend_top)
fig9.update_xaxes(title_text="Epoch")
fig9.update_yaxes(title_text="F1 Score", range=[0, 1.0])
fig9.write_html(str(charts_dir / "chart09_f1_curve.html"))
print(f"  ✓ custom_charts/chart09_f1_curve.html")

# ── Chart 10: Absolute Gap vs Both Baselines ─────────────────
_rgb         = [0.8946, 0.5255, 0.9210, 0.8245]
_gap_rgb     = [round(f - b, 4) for f, b in zip(_fusion, _rgb)]
_gap_thermal = [round(f - b, 4) for f, b in zip(_fusion, _thermal)]
fig10 = go.Figure()
fig10.add_trace(go.Bar(name='vs RGB Baseline',     x=_labels, y=_gap_rgb,
                       text=[f'{v:+.4f}' for v in _gap_rgb], textposition='outside',
                       marker_color=['#ef553b' if v < 0 else '#00cc96' for v in _gap_rgb]))
fig10.add_trace(go.Bar(name='vs Thermal Baseline', x=_labels, y=_gap_thermal,
                       text=[f'{v:+.4f}' for v in _gap_thermal], textposition='outside',
                       marker_color=['#636efa' if v < 0 else '#00cc96' for v in _gap_thermal]))
fig10.update_traces(cliponaxis=False)
fig10.add_hline(y=0, line_color='gray')
fig10.update_layout(barmode='group',
                    title="Simple Fusion Absolute Gap vs Both Baselines",
                    legend=legend_top)
fig10.update_xaxes(title_text="Metric")
fig10.update_yaxes(title_text="Absolute Difference",
                   range=[min(_gap_thermal) - 0.05, 0.05])
fig10.write_html(str(charts_dir / "chart10_absolute_gap.html"))
print(f"  ✓ custom_charts/chart10_absolute_gap.html")

# ── Chart 11: Heatmap ────────────────────────────────────────
import plotly.figure_factory as ff
_hm_models  = ["RGB Baseline", "Thermal Baseline", "Simple Fusion 50/50"]
_hm_metrics = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"]
_z = [
    [0.8946, 0.5255, 0.9210, 0.8245],
    [0.9625, 0.6600, 0.9597, 0.9016],
    [FINAL_MAP50, FINAL_MAP5095, FINAL_PREC, FINAL_REC],
]
_text = [[f'{v:.4f}' for v in row] for row in _z]
fig11 = ff.create_annotated_heatmap(z=_z, x=_hm_metrics, y=_hm_models,
                                     annotation_text=_text,
                                     colorscale='RdYlGn', showscale=True)
fig11.update_layout(title="Metrics Heatmap — All Models at a Glance")
fig11.update_xaxes(title_text="Metric")
fig11.update_yaxes(title_text="Model")
fig11.write_html(str(charts_dir / "chart11_heatmap.html"))
print(f"  ✓ custom_charts/chart11_heatmap.html")

# ── Chart 12: Precision & Recall Subplots ────────────────────
from plotly.subplots import make_subplots
fig12 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                      subplot_titles=("Precision Over Epochs", "Recall Over Epochs"),
                      vertical_spacing=0.12)
fig12.add_trace(go.Scatter(x=epochs, y=precision, mode='lines', name='Precision',
                           fill='tozeroy', fillcolor='rgba(99,110,250,0.08)'), row=1, col=1)
fig12.add_trace(go.Scatter(x=epochs, y=recall, mode='lines', name='Recall',
                           fill='tozeroy', fillcolor='rgba(239,85,59,0.08)'), row=2, col=1)
for row in [1, 2]:
    fig12.add_vline(x=BEST_EPOCH, line_dash='dash', line_color='gray',
                    annotation_text='Best (ep 54)',
                    annotation_position='top right', row=row, col=1)
fig12.update_yaxes(title_text="Precision", range=[0, 1.05], row=1, col=1)
fig12.update_yaxes(title_text="Recall",    range=[0, 1.05], row=2, col=1)
fig12.update_xaxes(title_text="Epoch", row=2, col=1)
fig12.update_layout(title="Precision & Recall Over Training (74 Epochs)", legend=legend_top)
fig12.write_html(str(charts_dir / "chart12_precision_recall_subplots.html"))
print(f"  ✓ custom_charts/chart12_precision_recall_subplots.html")

# ────────────────────────────────────────────────────────────
# 4. METRICS — CSV + JSON
#    └── metrics/
#        ├── training_log.csv
#        ├── final_metrics.json
#        ├── all_baselines_comparison.csv
#        └── f1_per_epoch.csv
# ────────────────────────────────────────────────────────────
metrics_dir = SAVE_ROOT / "metrics"
metrics_dir.mkdir(exist_ok=True)

# training_log.csv
with open(metrics_dir / "training_log.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch","box_loss","cls_loss","dfl_loss",
                     "precision","recall","mAP50","mAP50_95","f1"])
    for i, row in enumerate(EPOCH_LOG):
        f1_val = round(2 * row[4] * row[5] / (row[4] + row[5]), 4) if (row[4] + row[5]) > 0 else 0
        writer.writerow(list(row) + [f1_val])
print(f"  ✓ metrics/training_log.csv  ({len(EPOCH_LOG)} epochs + F1 column)")

# final_metrics.json
json.dump({
    "experiment"      : "Simple Fusion V2 — 50/50 Pixel Blend",
    "timestamp"       : timestamp,
    "dataset"         : "LLVIP",
    "model"           : "YOLOv8n",
    "fusion_type"     : "pixel-level weighted average",
    "rgb_weight"      : 0.5,
    "thermal_weight"  : 0.5,
    "train_images"    : 12025,
    "val_images"      : 3463,
    "epochs_trained"  : epochs_trained,
    "best_epoch"      : best_epoch,
    "early_stopping"  : True,
    "patience"        : 20,
    "training_time_h" : 3.241,
    "results": {
        "mAP50"     : FINAL_MAP50,
        "mAP50_95"  : FINAL_MAP5095,
        "precision" : FINAL_PREC,
        "recall"    : FINAL_REC,
        "f1"        : round(2 * FINAL_PREC * FINAL_REC / (FINAL_PREC + FINAL_REC), 4),
    },
    "baselines": {
        "RGB_Baseline"     : {"mAP50":0.8946,"mAP50_95":0.5255,"precision":0.9210,"recall":0.8245},
        "Thermal_Baseline" : {"mAP50":0.9625,"mAP50_95":0.6600,"precision":0.9597,"recall":0.9016},
    }
}, open(metrics_dir / "final_metrics.json", "w"), indent=4)
print(f"  ✓ metrics/final_metrics.json")

# all_baselines_comparison.csv
with open(metrics_dir / "all_baselines_comparison.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Model","mAP@0.5","mAP@0.5:0.95","Precision","Recall","F1"])
    writer.writerow(["RGB Baseline",        0.8946, 0.5255, 0.9210, 0.8245,
                     round(2*0.9210*0.8245/(0.9210+0.8245), 4)])
    writer.writerow(["Thermal Baseline",    0.9625, 0.6600, 0.9597, 0.9016,
                     round(2*0.9597*0.9016/(0.9597+0.9016), 4)])
    writer.writerow(["Simple Fusion 50/50", FINAL_MAP50, FINAL_MAP5095,
                     FINAL_PREC, FINAL_REC,
                     round(2*FINAL_PREC*FINAL_REC/(FINAL_PREC+FINAL_REC), 4)])
print(f"  ✓ metrics/all_baselines_comparison.csv")

# ────────────────────────────────────────────────────────────
# 5. YAML CONFIG
# ────────────────────────────────────────────────────────────
yaml_path = WORK / "simple_fusion_v2.yaml"
if yaml_path.exists():
    shutil.copy(str(yaml_path), str(SAVE_ROOT / "simple_fusion_v2.yaml"))
    print(f"  ✓ simple_fusion_v2.yaml")
else:
    print(f"  ⚠ YAML not found")

# ────────────────────────────────────────────────────────────
# 6. README
# ────────────────────────────────────────────────────────────
FINAL_F1 = round(2 * FINAL_PREC * FINAL_REC / (FINAL_PREC + FINAL_REC), 4)
readme = f"""# Simple Fusion V2 — Results Package
Generated : {timestamp}

## Experiment
- Method         : 50/50 pixel-level blend of RGB + Thermal (LLVIP)
- Model          : YOLOv8n (standard, no architecture changes)
- Dataset        : LLVIP — 12,025 train / 3,463 val images
- Epochs trained : {epochs_trained}  (early stopping patience=20)
- Best epoch     : {best_epoch}
- Training time  : 3.241 hours

## Final Results (best.pt — full validation)
| Metric        | Value   |
|---------------|---------|
| mAP@0.5       | {FINAL_MAP50:.4f}  |
| mAP@0.5:0.95  | {FINAL_MAP5095:.4f}  |
| Precision     | {FINAL_PREC:.4f}  |
| Recall        | {FINAL_REC:.4f}  |
| F1 Score      | {FINAL_F1:.4f}  |

## Baseline Comparison
| Model                | mAP@0.5 | mAP@0.5:0.95 | Precision | Recall | F1     |
|----------------------|---------|--------------|-----------|--------|--------|
| RGB Baseline         | 0.8946  | 0.5255       | 0.9210    | 0.8245 | 0.8701 |
| Thermal Baseline     | 0.9625  | 0.6600       | 0.9597    | 0.9016 | 0.9297 |
| Simple Fusion 50/50  | {FINAL_MAP50:.4f}  | {FINAL_MAP5095:.4f}  | {FINAL_PREC:.4f}  | {FINAL_REC:.4f}  | {FINAL_F1:.4f}  |

## Folder Structure
simple_fusion_v2_results/
├── model_weights/
│   ├── best.pt                        ← best checkpoint (epoch {best_epoch})
│   └── last.pt                        ← final checkpoint (plateau)
├── training_plots/                    ← YOLOv8 auto-generated plots
│   ├── results.png
│   ├── confusion_matrix.png
│   ├── confusion_matrix_normalized.png
│   ├── PR_curve.png
│   ├── F1_curve.png
│   ├── P_curve.png
│   ├── R_curve.png
│   ├── labels.jpg
│   ├── labels_correlogram.jpg
│   ├── val_batch0_pred.jpg
│   ├── val_batch1_pred.jpg
│   └── val_batch2_pred.jpg
├── custom_charts/                     ← Interactive Plotly charts (open in browser)
│   ├── chart01_loss_curves.html
│   ├── chart02_val_metrics.html
│   ├── chart03_map_gap.html
│   ├── chart04_model_comparison_bar.html
│   ├── chart05_radar.html
│   ├── chart06_pr_scatter.html
│   ├── chart07_pct_gap_thermal.html
│   ├── chart08_best_vs_final.html
│   ├── chart09_f1_curve.html
│   ├── chart10_absolute_gap.html
│   ├── chart11_heatmap.html
│   └── chart12_precision_recall_subplots.html
├── metrics/
│   ├── training_log.csv               ← epoch-by-epoch log + F1 column
│   ├── final_metrics.json             ← full summary with baselines + F1
│   └── all_baselines_comparison.csv   ← 3-model comparison with F1
├── simple_fusion_v2.yaml              ← dataset config (reproducibility)
└── README.md
"""
with open(SAVE_ROOT / "README.md", "w") as f:
    f.write(readme)
print(f"  ✓ README.md")

# ────────────────────────────────────────────────────────────
# 7. CREATE ZIP
# ────────────────────────────────────────────────────────────
zip_output = Path("/kaggle/working") / f"simple_fusion_v2_results_{timestamp}"
shutil.make_archive(str(zip_output), "zip", str(WORK), "simple_fusion_v2_results")
zip_path = Path(str(zip_output) + ".zip")
zip_size = zip_path.stat().st_size / (1024 * 1024)

print("\n" + "=" * 60)
print("  ALL DONE — DOWNLOAD YOUR RESULTS")
print("=" * 60)
print(f"  ZIP file : {zip_path.name}")
print(f"  ZIP size : {zip_size:.1f} MB")
print(f"  Location : {zip_path}")
print("=" * 60)
print("""
  Folder contents:
  ├── model_weights/     (best.pt + last.pt)
  ├── training_plots/    (12 YOLO auto-plots)
  ├── custom_charts/     (12 interactive HTML charts)
  ├── metrics/           (CSV + JSON logs)
  ├── simple_fusion_v2.yaml
  └── README.md

  → Kaggle Output panel → download ZIP directly.
""")

  SIMPLE FUSION V2 — PACKAGING RESULTS
  ✓ model_weights/best.pt
  ✓ model_weights/last.pt
  ✓ training_plots/results.png
  ✓ training_plots/confusion_matrix.png
  ✓ training_plots/confusion_matrix_normalized.png
  ✓ training_plots/labels.jpg
  ✓ training_plots/val_batch0_pred.jpg
  ✓ training_plots/val_batch1_pred.jpg
  ✓ training_plots/val_batch2_pred.jpg
  ✓ custom_charts/chart01_loss_curves.html
  ✓ custom_charts/chart02_val_metrics.html
  ✓ custom_charts/chart03_map_gap.html
  ✓ custom_charts/chart04_model_comparison_bar.html
  ✓ custom_charts/chart05_radar.html
  ✓ custom_charts/chart06_pr_scatter.html
  ✓ custom_charts/chart07_pct_gap_thermal.html
  ✓ custom_charts/chart08_best_vs_final.html
  ✓ custom_charts/chart09_f1_curve.html
  ✓ custom_charts/chart10_absolute_gap.html
  ✓ custom_charts/chart11_heatmap.html
  ✓ custom_charts/chart12_precision_recall_subplots.html
  ✓ metrics/training_log.csv  (74 epochs + F1 column)
  ✓ metrics/final_metrics.json
  ✓ metrics/all_baselines_c